# 🛣️ YOLOv8 Dual-Model Production Notebook v3
## Pothole/Crack Segmentation + Pavement Classification

---

**Features:**
- **Dual Model Support**: Runs Pothole Segmentation AND Pavement Classification simultaneously.
- **GPS Mapping**: Syncs Physics Toolbox logs.
- **Area Calculation**: Estimates defect size in m².
- **Combined Output**: CSV includes `pavement_type` for every detection.

In [1]:
# @title 1️⃣ Setup & Configuration
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Enable GPU: Runtime → Change runtime type → GPU")

# @markdown ### ⚙️ Active Features
USE_GPS = True # @param {type:"boolean"}
USE_CALIBRATION = False # @param {type:"boolean"}
GENERATE_VIDEO = True # @param {type:"boolean"}

# @markdown ### 🔧 Inference Settings
CONFIDENCE_THRESHOLD = 0.25
FRAME_SKIP = 1 # @param {type:"integer"}
IMG_SIZE = 640

# --- AREA CALCULATION - FALLBACK ---
ROAD_WIDTH_M = 3.5    # Standard lane width
CAMERA_HEIGHT_M = 1.5 # Approx camera height
# -----------------------

print(f"✅ Settings Loaded | Skip: {FRAME_SKIP} | Video: {GENERATE_VIDEO}")

!pip install -q ultralytics opencv-python pandas numpy matplotlib tqdm


CUDA: True
GPU: Tesla T4
✅ Settings Loaded | Skip: 1 | Video: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.8 MB/s eta 0:00:00


In [2]:
# @title 2️⃣ Define File Paths
# @markdown Enter the absolute paths to your files (e.g. uploaded to Colab or mounted Drive).

# @markdown ### 📹 Input Video
VIDEO_PATH = "/content/Trial_1.MOV" # @param {type:"string"}

# @markdown ### 🧠 Models
SEG_MODEL_PATH = "/content/combined_seg_best.pt" # @param {type:"string"}
CLS_MODEL_PATH = "/content/road_classification_best.pt" # @param {type:"string"}

# @markdown ### 📍 Optional Files
GPS_LOG_PATH = "/content/UPLB_Kanan_to_Kaliwa_2.csv" # @param {type:"string"}
CALIB_IMAGE_PATH = "/content/A4_paper.jpg" # @param {type:"string"}

import os
if not os.path.exists(VIDEO_PATH):
    print(f"⚠️ Warning: Video not found at {VIDEO_PATH}")
else:
    print(f"✅ Video found: {VIDEO_PATH}")


✅ Video found: /content/Trial_1.MOV


##3️⃣ Load GPS Data

In [4]:
import pandas as pd

gps_data = None
video_start_time = None

if USE_GPS and os.path.exists(GPS_LOG_PATH):
    try:
        df = pd.read_csv(GPS_LOG_PATH)
        df.columns = df.columns.str.strip()
        col_map = {col: col.lower() for col in df.columns}
        df = df.rename(columns=col_map)
        df['time'] = pd.to_datetime(df['time'], format='mixed', utc=True)
        df = df.sort_values('time')
        gps_data = df
        video_start_time = gps_data['time'].iloc[0]
        print(f"✅ Loaded {len(gps_data)} GPS points")
    except Exception as e:
        print(f"⚠️ Error loading GPS: {e}")
else:
    print("⏩ GPS Skipped or Not Found")


✅ Loaded 48114 GPS points


## 5️⃣ Load Models (Seg + Cls)

In [6]:
from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

# Load Segmentation
if os.path.exists(SEG_MODEL_PATH):
    seg_model = YOLO(SEG_MODEL_PATH)
    seg_model.to(device)
    SEG_CLASSES = seg_model.names
    print(f"✅ Seg Model Loaded: {SEG_CLASSES}")
else:
    print(f"❌ Seg Model NOT found at {SEG_MODEL_PATH}")

# Load Classification
if os.path.exists(CLS_MODEL_PATH):
    cls_model = YOLO(CLS_MODEL_PATH)
    cls_model.to(device)
    print("✅ Cls Model Loaded")
else:
    print(f"❌ Cls Model NOT found at {CLS_MODEL_PATH}")


Using Device: cuda
✅ Seg Model Loaded: {0: 'Cracks', 1: 'Potholes'}
✅ Cls Model Loaded


## 5️⃣ 📏 Calibration (Smart Area)

In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

H_matrix = None

if USE_CALIBRATION and os.path.exists(CALIB_IMAGE_PATH):
    print("✅ Calibration Image Loaded. Hover to find corners.")
    img = cv2.imread(CALIB_IMAGE_PATH)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.show()

    # --- MANUAL INPUT ---
    # Order: Top-Left, Top-Right, Bottom-Right, Bottom-Left
    PTS_SRC = np.array([
        [0, 0],   # REPLACE with actual Top-Left (x, y)
        [100, 0], # REPLACE
        [100, 100], # REPLACE
        [0, 100]    # REPLACE
    ], dtype=np.float32)

    PTS_DST = np.array([
        [0, 0], [0.21, 0], [0.21, 0.297], [0, 0.297]
    ], dtype=np.float32)

    H_matrix = cv2.getPerspectiveTransform(PTS_SRC, PTS_DST)
    print("✅ Homography Matrix Calculated!")
else:
    print("⏩ Calibration Skipped (Using Fallback PPM)")


⏩ Calibration Skipped (Using Fallback PPM)


## 6️⃣ Process Video (Dual Model Inference)

In [ ]:
import numpy as np
from tqdm import tqdm
import shutil

# --- AREA FUNCTIONS ---
def calculate_area_homography(mask, H):

    # FIX: Convert float mask (0.0-1.0) to binary text (0 or 255)
    mask_uint8 = (mask > 0.5).astype(np.uint8) * 255 

    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    total_area_m2 = 0.0
    for cnt in contours:
        if len(cnt) < 3: continue
        cnt = cnt.reshape(-1, 1, 2).astype(np.float32)
        dst_cnt = cv2.perspectiveTransform(cnt, H)
        area = cv2.contourArea(dst_cnt)
        total_area_m2 += area
    return total_area_m2

def calculate_area_simple(mask, width, height, road_width_m, camera_height_m):
    pixels_per_meter = (width / road_width_m + height / (camera_height_m * 1.5)) / 2
    pixel_count = np.sum(mask > 0)
    return pixel_count / (pixels_per_meter ** 2)

# --- SETUP ---
video_path = VIDEO_PATH
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"❌ Error: Could not open video at {video_path}")
    total_frames = 0
    fps = 30
    width = 640
    height = 640
else:
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

os.makedirs('results/temp_frames', exist_ok=True)
track_history = {} # ID -> List of {area, conf, frame_idx, ...}
frame_idx = 0
processed = 0

# --- VIDEO WRITER SETUP ---
video_out = None
if GENERATE_VIDEO and cap.isOpened():
    output_video_path = 'results/inference_video.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    print(f"🎥 Recording Video to {output_video_path}...")

pbar = tqdm(total=total_frames // FRAME_SKIP, desc="Tracking & Buffering")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    if frame_idx % FRAME_SKIP == 0:
        
        # 1. Classification (Road Type + Probabilities)
        pavement_type = "Unknown"
        probs_text = [] 
        
        if cls_model:
            cls_res = cls_model.predict(frame, verbose=False)[0]
            top1_idx = cls_res.probs.top1
            pavement_type = cls_res.names[top1_idx]
            
            # Extract Probabilities
            for i, score in enumerate(cls_res.probs.data.tolist()):
                name = cls_res.names[i]
                probs_text.append(f"{name}: {score:.2f}")

        # 2. Tracking (ByteTrack)
        seg_res = seg_model.track(frame, conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE, persist=True, tracker="bytetrack.yaml", verbose=False)[0]

        # 3. GPS Logic
        lat, lon = None, None
        if gps_data is not None and video_start_time is not None:
            video_time = video_start_time + pd.to_timedelta(frame_idx / fps, unit='s')
            closest_idx = (gps_data['time'] - video_time).abs().idxmin()
            gps_row = gps_data.loc[closest_idx]
            lat, lon = float(gps_row['latitude']), float(gps_row['longitude'])

        # 4. Processing

        annotated_frame = frame.copy()
        
        # --- DRAW VISUALS (ALWAYS) ---
        y_offset = 40
        
        for line in probs_text:
            cv2.putText(annotated_frame, line, (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            y_offset += 25
        
        # Draw Final Decision
        cv2.putText(annotated_frame, f"Road: {pavement_type}", (20, y_offset + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        if seg_res.boxes is not None and seg_res.boxes.id is not None:
            boxes = seg_res.boxes.xyxy.cpu().numpy()
            track_ids = seg_res.boxes.id.cpu().numpy().astype(int)
            scores = seg_res.boxes.conf.cpu().numpy()
            classes = seg_res.boxes.cls.cpu().numpy().astype(int)
            masks = seg_res.masks.data.cpu().numpy() if seg_res.masks is not None else None
            
            for i, (box, track_id, score, cls) in enumerate(zip(boxes, track_ids, scores, classes)):
                x1, y1, x2, y2 = box.astype(int)
                class_name = SEG_CLASSES[cls]

                # Area Calc (Smart / Fallback)
                area_m2 = 0.0
                if masks is not None and i < len(masks):
                    mask = cv2.resize(masks[i], (width, height))
                    try:
                        if 'H_matrix' in globals() and H_matrix is not None:
                            area_m2 = calculate_area_homography(mask, H_matrix)
                        else:
                            area_m2 = calculate_area_simple(mask, width, height, ROAD_WIDTH_M, CAMERA_HEIGHT_M)
                    except:
                        pass

                # Visualization
                # --- VISUALIZATION (Red Mask + Red Box) ---
                color = (0, 0, 255) # Red (BGR)
                opacity = 0.5       # Visible but partially transparent
                
                # A. Draw Transparent Red Mask (No Outline)
                if masks is not None and i < len(masks):
                    mask_resize = cv2.resize(masks[i], (width, height))
                    mask_bool = mask_resize > 0.5
                    
                    roi = annotated_frame[mask_bool]
                    blended = roi.astype(float) * (1 - opacity) + np.array(color) * opacity
                    annotated_frame[mask_bool] = blended.astype(np.uint8)

                # B. Draw Bounding Box
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                
                # C. Clean Label: "Pothole 0.114m2"
                label = f"{class_name} {area_m2:.3f}m2" 
                cv2.putText(annotated_frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                
                # Buffer Logic
                if track_id not in track_history:
                    track_history[track_id] = []

                temp_filename = f"temp_{track_id}_{frame_idx}.jpg"
                temp_path = f"results/temp_frames/{temp_filename}"

                # Save temp image for Selection
                cv2.imwrite(temp_path, annotated_frame)

                track_history[track_id].append({
                    'frame_idx': frame_idx,
                    'area_m2': area_m2,
                    'confidence': float(score),
                    'temp_path': temp_path,
                    'data': { # Final Data Structure
                        'track_id': track_id,
                        'class': class_name,
                        'confidence_score': float(score),
                        'pavement_type': pavement_type,
                        'area_m2': area_m2,
                        'latitude': lat,
                        'longitude': lon,
                        'image_path': f'pothole_id_{track_id}.jpg',
                        'box_width': int(x2 - x1),
                        'box_height': int(y2 - y1),
                        'box_center_y': int((y1 + y2) / 2)
                    }
                })

            if video_out:
                video_out.write(annotated_frame)
        else:
            if video_out:
                annotated_frame = frame.copy()
                cv2.putText(annotated_frame, f"Road: {pavement_type}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
                video_out.write(annotated_frame)

        processed += 1
        pbar.update(1)

    frame_idx += 1

cap.release()
if video_out:
    video_out.release()
pbar.close()
print(f"\n✅ Buffer Complete. Analyzing {len(track_history)} unique tracks...")


🎥 Recording Video to results/inference_video.mp4...



Tracking & Buffering:   0%|          | 0/14060 [01:43<?, ?it/s]

Tracking & Buffering:   5%|▌         | 713/14060 [01:26<27:04,  8.22it/s]

In [ ]:
# @title 7️⃣ Save & Download Results
from google.colab import files
import shutil
import pandas as pd
import os

# Ensure output directory exists
os.makedirs('results/frames', exist_ok=True)

final_detections = []
print("Selecting Representative Frames (Closest to Average Area)...")

for track_id, detections in track_history.items():
    if not detections: continue
    high_conf = [d for d in detections if d['confidence'] > 0.4]
    candidates = high_conf if len(high_conf) > 0 else detections
    avg_area = np.mean([d['area_m2'] for d in candidates])
    best_detection = min(candidates, key=lambda x: abs(x['area_m2'] - avg_area))

    final_path = f"results/frames/pothole_id_{track_id}.jpg"
    shutil.copy(best_detection['temp_path'], final_path)
    final_detections.append(best_detection['data'])

shutil.rmtree('results/temp_frames', ignore_errors=True)
print(f"✅ Selection Complete. Saved {len(final_detections)} unique defects.")

if len(final_detections) > 0:
    df = pd.DataFrame(final_detections)
    df.to_csv('results/detections.csv', index=False)
    print("📊 Pavement Breakdown:")
    print(df['pavement_type'].value_counts())

    print("📦 Zipping results...")
    shutil.make_archive('final_results', 'zip', 'results')
    files.download('final_results.zip')
    print("🎉 Downloaded final_results.zip")
else:
    print("⚠️ No defects found to save.")
